# Young-Adult Interstate Migration: Policy Incentives, Social Networks, and Capacity Feedback — V8
## A spatial agent-based model with migration chains, housing/job constraints, and robustness tests

**Core question**

> How strong must a geographically targeted housing-and-employment policy be to overcome young adults' social attachment, and when do migration chains and local capacity constraints amplify or offset that response?

This revision adds two forms of robustness requested in the model review: **social-threshold sensitivity** and **network-homophily sensitivity**. It also adds a lagged **rent response to excess housing demand** so housing pressure affects next-year migration utility rather than appearing only as a hard capacity rejection.

The former `happiness` measure is renamed **social utility index**. It is a transparent model-internal composite score, not a validated subjective-well-being survey measure. Its role is diagnostic rather than a real-world welfare ranking.

**V7 review-closure revision.** This version closes the remaining high-priority review comments: mover-level unit consistency, a soft housing-capacity mechanism with lagged rent feedback, a main-horizon ACS move-rate calibration, a wider policy-intensity sweep, an external-competition scope audit using ACS PUMS, a Tulsa Remote monetary-reference diagnostic, and an explicit BRFSS coverage audit. Optional full geography expansion to Oklahoma/West Virginia is deliberately not mixed into the main eight-state model because it would require a new state-space and IRS-route recalibration; it is documented as external-validation future work rather than presented as completed evidence.


### Reproducible results build

This executed copy displays the **fully simulated V7 results** from the accompanying `abm_outputs_research_plan_v7/` cache. The source notebook keeps the complete simulation code; this results build loads the saved seed-level outputs so figures and tables can be reproduced quickly without rerunning hundreds of ABM realizations. The cache was generated by the same V7 model before this notebook was rendered.

In [ ]:
import json
import math
import os
import warnings
import zipfile
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.stats import t as student_t

try:
    from joblib import Parallel, delayed
    JOBLIB_AVAILABLE = True
except Exception:
    JOBLIB_AVAILABLE = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 170)

RANDOM_SEED = 42
BASE_YEAR = 2026

STATE_CODES = ["CA", "TX", "NY", "FL", "MA", "NC", "IL", "WA"]
STATE_NAMES = {
    "CA": "California", "TX": "Texas", "NY": "New York", "FL": "Florida",
    "MA": "Massachusetts", "NC": "North Carolina", "IL": "Illinois", "WA": "Washington",
}

# Full research design. Reduce these values only for debugging.
BASELINE_AGENTS = 1000
BASELINE_YEARS = 20
BASELINE_SEEDS = list(range(101, 109))

EXPERIMENT_AGENTS = 600
EXPERIMENT_YEARS = 12
CROSS_SEEDS = list(range(201, 209))
SWEEP_SEEDS = list(range(301, 309))
SHOCK_SEEDS = list(range(401, 407))
SENSITIVITY_SEEDS = list(range(501, 505))
CALIBRATION_SEEDS = (901, 902, 903, 904, 905)

N_JOBS = 1  # deterministic final execution; avoids process-backend stalls in notebook kernels
N_AGENTS = BASELINE_AGENTS
N_YEARS = BASELINE_YEARS

POLICY_COLS = [
    "housing_policy",
    "employment_incentive",
    "migration_incentive",
]

SOCIAL_UTILITY_WEIGHTS = {
    "peer": 0.30,
    "coorigin": 0.20,
    "home": 0.10,
    "employment": 0.20,
    "income": 0.10,
    "affordability": 0.10,
}

INTERACTION_COLORS = {
    "weak": "#4C72B0",
    "medium": "#DD8452",
    "strong": "#55A868",
}



def mean_t_ci(series):
    x = np.asarray(pd.Series(series).dropna(), dtype=float)
    if len(x) == 0:
        return np.nan, np.nan
    mean = float(x.mean())
    if len(x) == 1:
        return mean, np.nan
    half = float(
        student_t.ppf(0.975, len(x) - 1)
        * x.std(ddof=1)
        / np.sqrt(len(x))
    )
    return mean, half


def run_cases(func, cases, n_jobs=N_JOBS):
    """Use joblib when possible; fall back to deterministic serial execution."""
    if JOBLIB_AVAILABLE and n_jobs > 1:
        try:
            return Parallel(n_jobs=n_jobs, backend="loky")(
                delayed(func)(*case) for case in cases
            )
        except Exception as exc:
            print("Parallel execution unavailable; using serial fallback:", type(exc).__name__)
    return [func(*case) for case in cases]

print("Environment ready.")

Environment ready.


## 1. Empirical state inputs

In [ ]:
def build_real_state_snapshot():
    raw = pd.DataFrame({
        "state": STATE_CODES,

        "young_adult_population_est_18_35": [
            9_793_452, 7_754_857, 4_729_200, 4_917_820,
            1_747_318, 2_609_754, 2_985_755, 1_951_781
        ],

        "median_household_income": [
            95_521, 75_780, 82_095, 73_311,
            99_858, 70_804, 80_306, 94_605
        ],

        "median_gross_rent_monthly": [
            1_992, 1_413, 1_561, 1_719,
            1_757, 1_245, 1_238, 1_731
        ],

        "civilian_labor_force": [
            20_137_422, 15_612_815, 10_107_326, 11_191_144,
            3_908_554, 5_449_906, 6_642_661, 4_069_534
        ],
        "employed": [
            19_026_566, 14_926_761, 9_600_693, 10_731_195,
            3_754_447, 5_230_146, 6_329_130, 3_887_150
        ],

        "bachelor_plus_share_25plus": [
            0.3748778859, 0.3424161436, 0.4063985602, 0.3491620625,
            0.4778163562, 0.3679334714, 0.3827775836, 0.4045106716
        ],

        "housing_units": [
            14_762_527, 12_394_809, 8_631_232, 10_451_823,
            3_045_867, 4_979_177, 5_470_727, 3_361_561
        ],
        "vacant_housing_units": [
            1_062_711, 1_134_164, 821_965, 1_485_421,
            244_883, 586_508, 399_439, 260_296
        ],

        "ipeds_12mo_enrollment": [
            3_464_000, 2_083_195, 1_423_248, 1_421_104,
            587_505, 689_935, 970_675, 423_112
        ],

        "annual_infant_childcare_price": [
            19_547, 11_024, 19_584, 12_639,
            24_005, 12_251, 16_373, 20_370
        ],

        "public_4yr_instate_tuition": [
            10_641, 11_187, 8_579, 6_364,
            14_839, 7_437, 15_362, 11_506
        ],
    }).set_index("state")

    raw["employment_rate_labor_force"] = (
        raw["employed"] / raw["civilian_labor_force"]
    )
    raw["vacancy_rate"] = (
        raw["vacant_housing_units"] / raw["housing_units"]
    )
    raw["ipeds_enrollment_per_young_adult"] = (
        raw["ipeds_12mo_enrollment"]
        / raw["young_adult_population_est_18_35"]
    )
    return raw

real_raw_data = build_real_state_snapshot()

In [ ]:
def mean_index(series):
    x = series.astype(float)
    return x / x.mean()

def prepare_state_data(raw):
    df = raw.copy()

    df["youth_pop_weight"] = mean_index(
        df["young_adult_population_est_18_35"]
    )
    df["wage_index"] = mean_index(df["median_household_income"])
    df["rent_index"] = mean_index(df["median_gross_rent_monthly"])
    df["job_index"] = mean_index(df["employment_rate_labor_force"])
    df["education_access_index"] = mean_index(
        df["ipeds_enrollment_per_young_adult"]
    )
    df["tuition_index"] = mean_index(df["public_4yr_instate_tuition"])
    df["childcare_cost_index"] = mean_index(
        df["annual_infant_childcare_price"]
    )
    df["housing_capacity_index"] = mean_index(df["vacancy_rate"])
    df["college_share_proxy"] = df["bachelor_plus_share_25plus"]

    return df

state_data = prepare_state_data(real_raw_data)

## 2. Spatial environment and IRS route calibration

In [ ]:
STATE_COORDS = {
    "CA": (36.8, -119.4),
    "TX": (31.0, -99.9),
    "NY": (42.9, -75.5),
    "FL": (27.8, -81.7),
    "MA": (42.3, -71.8),
    "NC": (35.5, -79.4),
    "IL": (40.0, -89.2),
    "WA": (47.4, -120.7),
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = (
        np.sin(dp / 2) ** 2
        + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))

distance_km = pd.DataFrame(
    index=STATE_CODES, columns=STATE_CODES, dtype=float
)

for s1 in STATE_CODES:
    for s2 in STATE_CODES:
        distance_km.loc[s1, s2] = haversine_km(
            *STATE_COORDS[s1], *STATE_COORDS[s2]
        )

distance_index = distance_km / distance_km.to_numpy().max()

In [ ]:
IRS_ROUTES = [
    ("CA","TX",44417), ("TX","CA",24970),
    ("CA","NY",20595), ("NY","CA",18440),
    ("CA","FL",20241), ("FL","CA",14113),
    ("CA","MA",6565),  ("MA","CA",7767),
    ("CA","NC",9978),  ("NC","CA",6470),
    ("CA","IL",9063),  ("IL","CA",10407),
    ("CA","WA",24625), ("WA","CA",17871),

    ("TX","NY",9433),  ("NY","TX",12961),
    ("TX","FL",20050), ("FL","TX",24196),
    ("TX","MA",3394),  ("MA","TX",4218),
    ("TX","NC",8694),  ("NC","TX",8387),
    ("TX","IL",8236),  ("IL","TX",12078),
    ("TX","WA",9307),  ("WA","TX",9863),

    ("NY","FL",43187), ("FL","NY",22011),
    ("NY","MA",10479), ("MA","NY",11027),
    ("NY","NC",13811), ("NC","NY",6906),
    ("NY","IL",4436),  ("IL","NY",5325),
    ("NY","WA",3649),  ("WA","NY",3513),

    ("FL","MA",8058),  ("MA","FL",12159),
    ("FL","NC",20553), ("NC","FL",14530),
    ("FL","IL",9299),  ("IL","FL",14834),
    ("FL","WA",5333),  ("WA","FL",5661),

    ("MA","NC",3483),  ("NC","MA",2213),
    ("NC","WA",2371),  ("WA","NC",2464),
    ("IL","WA",2734),  ("WA","IL",2158),
]

irs_flows = pd.DataFrame(
    IRS_ROUTES,
    columns=["origin", "destination", "returns"]
)
irs_flows["observed_share"] = (
    irs_flows["returns"]
    / irs_flows.groupby("origin")["returns"].transform("sum")
)

irs_flow_matrix = (
    irs_flows.pivot(
        index="origin", columns="destination", values="returns"
    )
    .reindex(index=STATE_CODES, columns=STATE_CODES)
)

for s in STATE_CODES:
    irs_flow_matrix.loc[s, s] = np.nan

## 6. Interaction-driven ABM

The model combines household migration choice, social-network influence, finite housing and jobs, demographic turnover, and state-level environmental feedback. Housing is now represented as a **hybrid quantity-price constraint**: capacity can still reject excess households in the current year, but the state-specific rejection rate raises next-year rents through `rent_rejection_elasticity`. This makes excess demand partially self-correcting instead of producing only a mechanical capacity wall.

In [ ]:
class PolicySocialABM:
    def __init__(
        self,
        state_df,
        policies,
        n_agents=EXPERIMENT_AGENTS,
        years=EXPERIMENT_YEARS,
        seed=RANDOM_SEED,
        params=None,
        interaction_strength=1.0,
        shocks=None,
        entrant_home_local=False,
        entrant_network_home_bias=False,
        network_homophily_share=0.0,
        record_events=True,
    ):
        self.base_state = state_df.copy()
        self.state = state_df.copy()
        self.policies = policies.copy()
        self.n_agents = int(n_agents)
        self.years = int(years)
        self.seed = int(seed)
        self.params = PARAMS.copy() if params is None else params.copy()
        self.interaction_strength = float(interaction_strength)
        self.shocks = {} if shocks is None else dict(shocks)
        self.entrant_home_local = bool(entrant_home_local)
        self.entrant_network_home_bias = bool(entrant_network_home_bias)
        self.network_homophily_share = float(np.clip(network_homophily_share, 0.0, 1.0))
        self.record_events = bool(record_events)

        self.params["beta_peer_network"] = PARAMS["beta_peer_network"] * self.interaction_strength
        self.params["beta_coorigin"] = PARAMS["beta_coorigin"] * self.interaction_strength
        self.params["beta_social_dissatisfaction"] = PARAMS["beta_social_dissatisfaction"] * self.interaction_strength

        self.agents, self.network = initialise_agents(
            state_df,
            n_agents=self.n_agents,
            seed=self.seed + 1000,
            network_k=self.params["network_k"],
            network_homophily_share=self.network_homophily_share,
        )
        self.network_k = self.params["network_k"]
        self.next_household_id = int(self.agents["household_id"].max() + 1)

        self.history = []
        self.move_history = []
        self.decision_rows = []
        self.mechanism_rows = []
        self.demography_rows = []
        self.snapshots = {}
        self.job_rejection_by_state = {}
        self.unfilled_jobs_by_state = {}
        self.housing_rejection_by_state = {}
        self.previous_counts = self._counts()
        self.initial_counts = self.previous_counts.copy()

        self._employment_step(-1)
        self._snapshot(BASE_YEAR)

    def _rng(self, code, t):
        return np.random.default_rng(np.random.SeedSequence([self.seed, int(code), int(t) + 100]))

    def _counts(self):
        return self.agents["state"].value_counts().reindex(STATE_CODES, fill_value=0).astype(float)

    def _state_indices(self):
        return pd.Categorical(self.agents["state"], categories=STATE_CODES).codes

    def _home_indices(self):
        return pd.Categorical(self.agents["home_state"], categories=STATE_CODES).codes

    def _shock_factors(self, t):
        if not self.shocks:
            return {
                "job_slot_multiplier": 1.0,
                "wage_multiplier": 1.0,
                "mortality_multiplier": 1.0,
                "migration_cost_multiplier": 1.0,
            }

        start = int(self.shocks.get("start", 0))
        duration = int(self.shocks.get("duration", 0))
        recovery = int(self.shocks.get("recovery_years", 0))

        if t < start:
            intensity = 0.0
        elif t < start + duration:
            intensity = 1.0
        elif recovery > 0 and t < start + duration + recovery:
            elapsed = t - (start + duration) + 1
            intensity = max(0.0, 1.0 - elapsed / (recovery + 1))
        else:
            intensity = 0.0

        def blend(key):
            target = float(self.shocks.get(key, 1.0))
            return 1.0 + intensity * (target - 1.0)

        return {
            "job_slot_multiplier": blend("job_slot_multiplier"),
            "wage_multiplier": blend("wage_multiplier"),
            "mortality_multiplier": blend("mortality_multiplier"),
            "migration_cost_multiplier": blend("migration_cost_multiplier"),
        }

    def _household_formation_step(self, t):
        a = self.agents
        rng = self._rng(15, t)
        hh = a["household_id"].to_numpy()
        sizes = a["household_id"].map(a["household_id"].value_counts()).to_numpy()
        eligible = (a["age"].to_numpy() >= 22) & (sizes == 1)
        states = a["state"].to_numpy()
        for state in STATE_CODES:
            candidates = np.where(eligible & (states == state))[0]
            if len(candidates) < 2:
                continue
            rng.shuffle(candidates)
            n_adults = int(len(candidates) * self.params["annual_pair_formation_rate"])
            n_adults -= n_adults % 2
            for i, j in candidates[:n_adults].reshape(-1, 2):
                hid = self.next_household_id
                self.next_household_id += 1
                a.loc[[i, j], "household_id"] = hid
                dep = max(int(a.loc[i, "dependent_children"]), int(a.loc[j, "dependent_children"]))
                if dep > 0:
                    a.loc[[i, j], "has_children"] = 1
                    a.loc[[i, j], "dependent_children"] = dep

    def _family_step(self, t):
        a = self.agents
        rng = self._rng(17, t)
        grouped = a.groupby("household_id", sort=False)
        hh_age = grouped["age"].mean()
        hh_children = grouped["dependent_children"].max()
        eligible = hh_age.index[
            (hh_age.to_numpy() >= 20)
            & (hh_age.to_numpy() <= 35)
            & (hh_children.to_numpy() < self.params["max_dependent_children"])
        ]
        if len(eligible) == 0:
            return 0
        ages = hh_age.loc[eligible].to_numpy()
        existing = hh_children.loc[eligible].to_numpy()
        age_profile = np.exp(-0.5 * ((ages - self.params["birth_age_center"]) / self.params["birth_age_spread"]) ** 2)
        p = np.clip(self.params["birth_peak_probability"] * age_profile * (1.0 - 0.18 * existing), 0, 0.09)
        selected = eligible[rng.random(len(eligible)) < p]
        for hid in selected:
            mask = a["household_id"] == hid
            current = int(a.loc[mask, "dependent_children"].max())
            a.loc[mask, "has_children"] = 1
            a.loc[mask, "dependent_children"] = min(current + 1, self.params["max_dependent_children"])
        return int(len(selected))


    def _employment_step(self, t):
        a = self.agents
        n = len(a)
        rng = self._rng(37, t)
        factors = self._shock_factors(t)
        employed = np.zeros(n, dtype=int)
        rejected_flag = np.zeros(n, dtype=int)
        states = a["state"].to_numpy()
        state_rejection = {}
        state_unfilled = {}

        for state in STATE_CODES:
            mask = states == state
            applicants = np.where(mask)[0]
            if len(applicants) == 0:
                state_rejection[state] = np.nan
                state_unfilled[state] = 0
                continue

            slot_rate = np.clip(
                0.80
                + 0.16 * (self.state.loc[state, "job_index"] - 1.0)
                + self.params["employment_policy_slots"] * self.policies.loc[state, "employment_incentive"],
                0.60,
                1.20,
            )
            slots = int(round(slot_rate * mask.sum() * factors["job_slot_multiplier"]))
            order = rng.permutation(applicants)
            remaining = slots
            for i in order:
                if remaining <= 0:
                    break
                p_match = logistic(
                    2.0
                    + 0.8 * (a.loc[i, "skill"] - 1.0)
                    + 0.30 * a.loc[i, "college"]
                    - 0.10 * min(a.loc[i, "unemployed_years"], 3)
                )
                if rng.random() < p_match:
                    employed[i] = 1
                    remaining -= 1

            rejected_here = applicants[employed[applicants] == 0]
            rejected_flag[rejected_here] = 1
            state_rejection[state] = len(rejected_here) / max(len(applicants), 1)
            state_unfilled[state] = max(remaining, 0)

        self.job_rejection_by_state[t] = state_rejection
        self.unfilled_jobs_by_state[t] = state_unfilled
        a["employed"] = employed
        a["job_rejected_last_year"] = rejected_flag
        a["unemployed_years"] = np.where(employed == 1, 0, a["unemployed_years"].to_numpy() + 1)

        wage = a["state"].map(self.state["wage_index"]).to_numpy() * factors["wage_multiplier"]
        base_wage = a["state"].map(self.base_state["wage_index"]).to_numpy()
        anchor = 0.65 * a["state"].map(self.base_state["median_household_income"]).to_numpy()
        income = (
            anchor * (wage / base_wage)
            * (0.72 + 0.28 * a["skill"].to_numpy())
            * (1.0 + 0.22 * a["college"].to_numpy())
            * np.where(employed == 1, 1.0, 0.34)
        )
        a["income"] = income * self._rng(41, t).lognormal(0, 0.04, n)


    def _snapshot(self, year):
        self.snapshots[year] = self.agents[[
            "agent_id", "state", "home_state", "age", "employed",
            "household_id", "has_children", "is_international"
        ]].copy()


